# open_macro_v03 phase0q — qc_research_object_store leg

This QuantConnect **Research** notebook reproduces the `local_python_pure` leg of the
open_macro_v03 phase0q reproducibility matrix inside `qc_research_object_store`. It:

1. reads the object-store **manifest key**, pulls every bundle object via the QuantBook
   Object Store, and verifies each object's `content_sha256` (drift refusal),
2. materializes the phase0q harness + `src` decision-path sources from the bundle, with
   a **fail-loud `src/db.py` stub** (offline-only; database access forbidden),
3. re-runs the local-leg computation — the decision chain plus the `baseline_100` and
   `compressed_50` sleeves at the **base 5bps minimum** (full cost grid / all four
   variants only if runtime permits),
4. recomputes the canonical logical hashes and compares them to the
   `expected_results_manifest.json` (exact hash match; 1e-12 float tolerance),
5. emits a **verdict JSON** back to the Object Store under the same immutable prefix.

**Governance (non-negotiable):** A5 stays **blocked**; `runtime_activation`,
`activation_allowed`, `allocator_publish`, `official_result` are all **false**;
`db_write_mode` is `none`; `status` is `candidate_not_approved`. A matching verdict is
reproducibility evidence only and grants no activation or approval. QC FRED / History is
intentionally absent from this notebook.

## 1. Object Store contract + manifest

Set `PHASE0Q_CLOUD_MANIFEST_KEY` (env or the committed `object_store_manifest_key.txt`)
to the immutable manifest key printed by the upload plan. The notebook reads ONLY the
Investintell bundle objects listed in that manifest.

In [ ]:
from pathlib import Path
import gzip
import hashlib
import importlib
import json
import os
import platform
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEFAULT_MANIFEST_KEY = ""  # filled from object_store_manifest_key.txt or env at run time
MANIFEST_KEY_FILE = Path("object_store_manifest_key.txt")
MANIFEST_KEY = os.environ.get("PHASE0Q_CLOUD_MANIFEST_KEY")
if not MANIFEST_KEY and MANIFEST_KEY_FILE.exists():
    MANIFEST_KEY = MANIFEST_KEY_FILE.read_text(encoding="utf-8").strip()
if not MANIFEST_KEY:
    MANIFEST_KEY = DEFAULT_MANIFEST_KEY
REQUIRE_QC_CLOUD = os.environ.get("PHASE0Q_REQUIRE_QC_CLOUD", "1").strip().lower() not in {"0", "false", "no"}
PLATFORM_TEXT = platform.platform()


def maybe_quantbook_object_store():
    try:
        from QuantConnect.Research import QuantBook
    except Exception:
        return None
    try:
        qb = QuantBook()
    except Exception:
        return None
    return getattr(qb, "object_store", getattr(qb, "ObjectStore", None))


object_store = maybe_quantbook_object_store()


def object_store_file_path(key):
    if object_store is None:
        return None
    if hasattr(object_store, "get_file_path"):
        return Path(object_store.get_file_path(key))
    if hasattr(object_store, "GetFilePath"):
        return Path(object_store.GetFilePath(key))
    raise RuntimeError("QuantConnect Object Store does not expose get_file_path/GetFilePath")


if not MANIFEST_KEY:
    raise RuntimeError("set PHASE0Q_CLOUD_MANIFEST_KEY or provide object_store_manifest_key.txt")

manifest_path = object_store_file_path(MANIFEST_KEY)
bundle_source = "quantbook_object_store" if manifest_path is not None else "local_bundle_fallback"
if manifest_path is None:
    # local fallback: manifest_key_file sibling bundle (used only for offline dry-runs)
    manifest_path = Path("_phase0q_cloud_bundle/object_store_manifest.json")

if REQUIRE_QC_CLOUD:
    if bundle_source != "quantbook_object_store":
        raise RuntimeError("QC cloud mode requires bundle_source=quantbook_object_store")
    if "WSL2" in PLATFORM_TEXT:
        raise RuntimeError(f"QC cloud mode cannot run on WSL2 platform: {PLATFORM_TEXT}")

manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
manifest["_local_bundle_dir"] = str(Path(manifest_path).parent)
assert manifest["bridge_scope"] == "qc_research_phase0q_reproducibility_only"
assert manifest["governance"]["A5"] == "blocked"
assert manifest["governance"]["runtime_activation"] is False
assert manifest["governance"]["activation_allowed"] is False
assert manifest["governance"]["official_result"] is False
assert manifest["governance"]["db_write_mode"] == "none"
manifest["object_store_prefix_immutable"]

## 2. Pull objects + verify content_sha256 (drift refusal)

Every object in `manifest["object_files"]` is fetched (or read from the local bundle
dir) and its SHA-256 must match `content_sha256`; a mismatch **aborts** the run.

In [ ]:
bundle_dir = Path(manifest["_local_bundle_dir"])


def _sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def resolve_object(rel_path, expected_sha):
    item = manifest["object_files"][rel_path]
    key = item["object_store_key"]
    path = object_store_file_path(key)
    if path is None:
        path = bundle_dir / item["relative_path"]
    actual = _sha256(path)
    if actual != expected_sha:
        raise RuntimeError(
            f"drift refusal: object {rel_path} sha {actual} != manifest {expected_sha}")
    return Path(path)


resolved = {}
for rel_path, item in sorted(manifest["object_files"].items()):
    resolved[rel_path] = resolve_object(rel_path, item["content_sha256"])
print(f"verified {len(resolved)} objects against manifest content_sha256")

## 3. Materialize harness + src sources (fail-loud db stub)

Each gzipped source is decompressed to its repo-relative path under the working tree so
`from harness.phase0q import runner` and the `src` decision-path imports resolve. The
real `src/db.py` (psycopg / network) is replaced by the bundle's **fail-loud stub**.

In [ ]:
def _materialize_gz(rel_gz_path, target_rel):
    src_path = resolved[rel_gz_path]
    text = gzip.decompress(Path(src_path).read_bytes()).decode("utf-8")
    target = PROJECT_ROOT / target_rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding="utf-8", newline="\n")
    # ensure package __init__ files exist for every parent package
    parent = target.parent
    while parent != PROJECT_ROOT and parent.name:
        init = parent / "__init__.py"
        if not init.exists():
            init.write_text("", encoding="utf-8")
        parent = parent.parent
    return target


# harness sources + src closure: strip the "code/" prefix and ".gz" suffix to recover
# the repo-relative path each object should live at.
for rel_path in sorted(resolved):
    if not rel_path.startswith("code/") or not rel_path.endswith(".gz"):
        continue
    target_rel = rel_path[len("code/"):-len(".gz")]
    _materialize_gz(rel_path, target_rel)

# fail-loud db stub is shipped as code/src/db.py.gz and materialized above; assert it
# refuses connections so the cloud leg can never touch a database.
import src.db as _db  # noqa: E402
assert _db.LOCK_REGIME_QUADRANT == 900_208
try:
    _db.connect()
    raise AssertionError("db stub must refuse connect()")
except RuntimeError as exc:
    assert "offline-only" in str(exc)
print("harness + src materialized; fail-loud db stub verified")

## 4. Re-run the local-leg computation

Reload the materialized harness and run the decision chain + `baseline_100` /
`compressed_50` sleeves at the base 5bps minimum, then the full grid if runtime permits.
The pack v2 canonical tables come from the bundle (`pack/*.json`).

In [ ]:
import datetime as dt  # noqa: E402

# reload in case a stale copy was imported earlier in the kernel
for _mod in ["harness.phase0q.runner", "harness.phase0q.grid", "harness.phase0q.sleeve",
             "harness.phase0q.decision", "harness.phase0q.metrics", "harness.phase0q.pit"]:
    if _mod in sys.modules:
        importlib.reload(sys.modules[_mod])

from harness.phase0q import runner  # noqa: E402

scenario = json.loads((bundle_dir / "scenario_config.json").read_text(encoding="utf-8"))
rc = scenario["run_config"]

# Reconstruct the FULL pack v2 tree from the verified pack/ objects so
# load_and_verify_pack (which digests every pack file) runs faithfully.
pack_root = bundle_dir / "_pack_root"
for rel_path in sorted(resolved):
    if not rel_path.startswith("pack/"):
        continue
    dest = pack_root / rel_path[len("pack/"):]
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(Path(resolved[rel_path]).read_bytes())

eod_rows = json.loads((pack_root / "data" / "canonical" / "eod_prices.json").read_text(encoding="utf-8"))

from harness.phase0q import sleeve, decision, grid  # noqa: E402

prices = sleeve.PriceFrame(eod_rows)
candidates = tuple(
    sleeve.SleeveParams(
        c["candidate_id"], c["growth_weight"], c["inflation_weight"],
        c["risk_tilt"], c["defensive_floor_delta_pp"], c["risk_cap_delta_pp"])
    for c in rc["candidates"]
)
primary_window = (dt.date.fromisoformat(rc["primary_window"][0]),
                  dt.date.fromisoformat(rc["primary_window"][1]))
stress_windows = tuple(
    {"window_id": w["window_id"], "start": dt.date.fromisoformat(w["start"]),
     "end": dt.date.fromisoformat(w["end"]), "coverage": w["coverage"]}
    for w in rc["stress_windows"]
)
config = runner.RunConfig(
    run_id=rc["run_id"], started_at=rc["started_at"], finished_at=rc["finished_at"],
    harness_commit=rc["harness_commit"], candidates=candidates,
    cost_grid=tuple(rc["cost_grid_bps"]), primary_window=primary_window,
    stress_windows=stress_windows,
)

# full local-leg run reproduces the committed metric_evidence_001 hashes.
run = runner.run_harness(pack_root, config)

# baseline_100 / compressed_50 sleeve measurement at base 5bps (min); full variant grid
# if runtime permits. This mirrors compression_grid_001 for the two named sleeves.
base_decisions = run["decisions"]
grid_results = grid.measure_grid_results(
    prices, base_decisions, sleeve.SleeveParams("baseline_current", 0.5, 0.5, 0.0, 0.0, 0.0))
sleeve_hashes = {
    vid: runner.stable_hash(runner.canonicalize(grid_results[vid]))
    for vid in ("baseline_100", "compressed_50")
}
run["result"]["output_logical_hashes"]

## 5. Recompute hashes + compare to expected + emit verdict

Compare the recomputed contract-v2 hashes against `expected_results_manifest.json`
(exact match; 1e-12 float tolerance). Emit the verdict JSON both locally (`results/`)
and back to the Object Store under the same immutable prefix.

In [ ]:
import math  # noqa: E402

expected = json.loads((bundle_dir / "expected_results_manifest.json").read_text(encoding="utf-8"))
exp_hashes = expected["output_logical_hashes"]
got_hashes = run["result"]["output_logical_hashes"]

per_hash = {k: {"expected": exp_hashes[k], "actual": got_hashes.get(k),
                "match": exp_hashes[k] == got_hashes.get(k)} for k in sorted(exp_hashes)}
exp_fp = expected["run_fingerprint"]
got_fp = run["result"]["run_fingerprint"]
exp_local = expected["execution_legs"]["local_python_pure"]["logical_hash"]
got_cloud = run["result"]["execution_legs"][0]["logical_hash"]

mismatches = [k for k, v in per_hash.items() if not v["match"]]
all_match = not mismatches and exp_fp == got_fp and exp_local == got_cloud

verdict = {
    "artifact_type": "phase0q_cloud_leg_verdict",
    "schema_version": 1,
    "execution_backend": "quantconnect_cloud_research" if bundle_source == "quantbook_object_store" else "local_research_fallback",
    "bundle_source": bundle_source,
    "object_store_prefix_immutable": manifest["object_store_prefix_immutable"],
    "harness_commit": manifest["harness_commit"],
    "input_pack_sha256": manifest["input_pack_sha256"],
    "contract_bundle_sha256": manifest["contract_bundle_sha256"],
    "run_fingerprint": got_fp,
    "output_logical_hashes": got_hashes,
    "execution_legs": {
        "qc_research_object_store": {"logical_hash": got_cloud, "status": "complete"},
    },
    "sleeve_logical_hashes": sleeve_hashes,
    "comparison": {
        "output_logical_hashes": per_hash,
        "run_fingerprint": {"expected": exp_fp, "actual": got_fp, "match": exp_fp == got_fp},
        "execution_leg_logical_hash": {
            "expected_local_python_pure": exp_local,
            "actual_qc_research_object_store": got_cloud,
            "match": exp_local == got_cloud},
        "float_tolerance": 1e-12,
        "mismatch_count": len(mismatches) + (0 if exp_fp == got_fp else 1) + (0 if exp_local == got_cloud else 1),
        "all_hashes_match": all_match,
    },
    "external_macro_access": False,
    "reproduced": all_match,
    "verdict": "reproduced" if all_match else "not_reproduced",
    "governance": manifest["governance"],
}

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)
verdict_bytes = (json.dumps(verdict, sort_keys=True, indent=2, ensure_ascii=False) + "\n").encode("utf-8")
(results_dir / "phase0q_cloud_verdict.json").write_bytes(verdict_bytes)

# emit back to Object Store under the same immutable prefix (best-effort in cloud).
verdict_key = manifest.get("verdict_key_template")
if object_store is not None and verdict_key:
    try:
        if hasattr(object_store, "save_bytes"):
            object_store.save_bytes(verdict_key, verdict_bytes)
        elif hasattr(object_store, "SaveBytes"):
            object_store.SaveBytes(verdict_key, list(verdict_bytes))
    except Exception as exc:
        print(f"warning: could not write verdict to object store: {exc}")

assert verdict["governance"]["A5"] == "blocked"
assert verdict["governance"]["runtime_activation"] is False
assert verdict["governance"]["official_result"] is False
assert verdict["external_macro_access"] is False
print("verdict:", verdict["verdict"], "| mismatches:", verdict["comparison"]["mismatch_count"])
if not all_match:
    raise AssertionError(f"cloud leg did not reproduce local hashes: {mismatches}")
verdict["comparison"]["all_hashes_match"]